# 本单元的本地版本（Apple Silicon / M 系列芯片）

本笔记本是 [unit3.ipynb](unit3.ipynb) 的 **macOS 本地版**，与 Colab 版的差异：

- 删掉 `apt-get` / `xvfb` / `pyvirtualdisplay` 虚拟屏 —— 这些是 Linux 专属的；ALE 的 `rgb_array` 渲染和 `moviepy` 写视频在 Mac 上都不需要虚拟屏，`ffmpeg` 由 `imageio-ffmpeg` 自带。
- 删掉 `gymnasium[accept-rom-license]` —— `ale-py>=0.9` 已自带 ROM。
- CUDA 换成 Apple 的 MPS：SB3 的 `--device auto` 只探测 CUDA，在 Mac 上会回落 CPU，所以训练命令显式传 `--device {DEVICE}`。
- `!python` 改成 `!{sys.executable}`，保证 `!` 调起的子进程用的是当前内核（项目 `.venv`）的解释器，而不是 PATH 上的某个 Python。
- `dqn.yml` 由 `%%writefile` 生成，不用手动上传；正式训练前加了一次 2 万步的冒烟测试。
- `rl_zoo3.load_from_hub` 的命令行版在 `huggingface-hub` 1.x 下会崩，改为在内核里调用同一个函数并打补丁（原因见该单元格注释）。
- 显式安装老版 `gym`，用于加载 Hub 上用 `gym` 训练的旧模型（Colab 镜像自带）。
- 去掉 Hugging Face Hub 上传流程：回放用 `rl_zoo3.record_video` 存成本地 mp4 并在笔记本里内联播放；从 Hub 下载模型保留（不需要登录）。

准备环境（项目根目录）：

```bash
uv sync
```

然后在 Jupyter 中选择项目 `.venv` 作为内核。

# 单元 3：使用 RL Baselines3 Zoo 玩雅达利游戏 (Atari) 👾 进阶深层 Q 网络 (Deep Q-Learning)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/thumbnail.jpg" alt="Unit 3 Thumbnail">

在本 Notebook 中，**你将训练一个玩《太空侵略者》(Space Invaders) 的 Deep Q-Learning 智能体**。我们将使用 [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo)，这是一个基于 [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/) 的训练框架，提供了用于训练、评估智能体、调整超参数、绘制结果和录制视频的脚本。

我们使用的是 [RL-Baselines-3 Zoo 集成的原生版本 Deep Q-Learning](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html)，不包含 Double-DQN、Dueling-DQN 和优先经验回放 (Prioritized Experience Replay) 等扩展。

⬇️ 下面是你**即将实现的目标**示例 ⬇️

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/ThomasSimonini/ppo-SpaceInvadersNoFrameskip-v4/resolve/main/replay.mp4" type="video/mp4"></video>

### 🎮 运行环境：

- [SpacesInvadersNoFrameskip-v4](https://gymnasium.farama.org/environments/atari/space_invaders/)

你可以在这里查看不同版本《太空侵略者》之间的区别 👉 https://gymnasium.farama.org/environments/atari/space_invaders/#variants

### 📚 强化学习库：

- [RL-Baselines3-Zoo](https://github.com/DLR-RM/rl-baselines3-zoo)

## 本 Notebook 的目标 🏆
在学习完本 Notebook 后，你将能够：
- 深入理解 **RL Baselines3 Zoo 的工作原理**。
- 能够**将你训练好的智能体和代码推送到 Hugging Face Hub**，并附带精彩的视频回放和评估分数 🔥。

## 本教程来自 深度强化学习课程 (Deep Reinforcement Learning Course)
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

在这个免费课程中，你将：

- 📖 系统地在**理论与实践**中学习深度强化学习。
- 🧑‍💻 学习**使用著名的深度强化学习库**，如 Stable Baselines3、RL Baselines3 Zoo、CleanRL 和 Sample Factory 2.0。
- 🤖 在**独特的环境中训练智能体**

查看更多课程大纲 👉 https://simoninithomas.github.io/deep-rl-course

别忘了**<a href="http://eepurl.com/ic5ZUD">订阅课程</a>**（我们会收集您的邮箱，以便在每个单元发布时向您发送链接，并提供有关挑战和更新的信息）。

保持联系的最佳方式是加入我们的 Discord 服务器，与社区和我们进行交流 👉🏻 https://discord.gg/ydHrjt3WP5

## 前置准备 🏗️
在开始本 Notebook 之前，你需要：

🔲 📚 **[通过阅读第 3 单元学习 Deep Q-Learning](https://huggingface.co/deep-rl-course/unit3/introduction)** 🤗

我们一直在努力改进我们的教程，因此**如果你在此 Notebook 中发现任何问题**，请在 [Github 仓库上提交 Issue](https://github.com/huggingface/deep-rl-class/issues)。

# 让我们训练一个玩《太空侵略者》 👾 的 Deep Q-Learning 智能体并将其上传到 Hub。

为了通过认证流程，你需要将训练好的模型推送到 Hub，并**获得 >= 200 的分数**。

要查看你的结果，请转到排行榜并找到你的模型，**结果计算公式：result = mean_reward - std of reward**

有关认证流程的更多信息，请查看此部分 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

# 依赖 📚

用项目根目录的 `uv sync` 安装（见开头），本地版没有安装单元格。会用到：

- `rl-zoo3`：训练 / 评估 / 录像脚本，连带装上 `stable-baselines3`、`sb3-contrib` 和 `torch`（macOS arm64 的 wheel 自带 MPS 支持）。
- `gymnasium[atari]`：即 `ale-py`，雅达利模拟器和 ROM。
- `opencv-python`：SB3 的 `AtariWrapper` 用它做灰度化和缩放。
- `moviepy`：SB3 的 `VecVideoRecorder` 用它写 mp4。
- `gym`（老版 OpenAI Gym）：Hub 上 `sb3` 组织的老模型（如 BeamRider）是用 `gym` 保存的，SB3 加载时要先 `import gym` 反序列化空间，再经 `shimmy` 转成 gymnasium。Colab 镜像预装了 `gym`，所以原版不需要显式安装。
- `requests`：`rl_zoo3.load_from_hub` 要 import 它，但 rl_zoo3 没声明，`huggingface-hub` 1.x 也不再附带。

> 不要用 `%pip install rl_zoo3`：它依赖的 `huggingface-sb3 3.0`（PyPI 最新版）声明了 `huggingface-hub~=0.8`，pip 会把项目的 hub 1.x 降回 0.x。`pyproject.toml` 里用 `[tool.uv] override-dependencies` 放行了这条过时的约束（它实际用到的 hub API 在 1.x 仍存在）。

## 导入包 📦

In [ ]:
import os
import sys
import glob

import torch
from IPython.display import Video

# SB3 的 device="auto" 只探测 CUDA，Mac 上要显式传 mps
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(sys.executable, DEVICE)

## 训练我们的 Deep Q-Learning 智能体玩《太空侵略者》 👾

要使用 RL-Baselines3-Zoo 训练智能体，我们只需要做两件事：

1. 创建一个名为 `dqn.yml` 的超参数配置文件，其中将包含我们的训练超参数。

这是一个模板示例：

```
SpaceInvadersNoFrameskip-v4:
  env_wrapper:
    - stable_baselines3.common.atari_wrappers.AtariWrapper
  frame_stack: 4
  policy: 'CnnPolicy'
  n_timesteps: !!float 1e7
  buffer_size: 100000
  learning_rate: !!float 1e-4
  batch_size: 32
  learning_starts: 100000
  target_update_interval: 1000
  train_freq: 4
  gradient_steps: 1
  exploration_fraction: 0.1
  exploration_final_eps: 0.01
  # 如果为 True，您需要停用 replay_buffer_kwargs 中的 handle_timeout_termination
  optimize_memory_usage: False
```

在这里我们可以看到：
- 我们使用 `Atari Wrapper` 预处理输入（帧缩减、灰度化、堆叠 4 帧）
- 我们使用 `CnnPolicy`，因为我们使用卷积层来处理游戏帧图像
- 我们训练 1000 万个 `n_timesteps`
- 经验回放缓冲区 (Memory Buffer) 的大小为 100000，即你保存并用来再次训练智能体的经验步数。

💡 本地版的耗时：看下面冒烟测试日志里的 `time/fps`，按 `1e6 / fps` 估算 1M 步的时间。本机 M5 / 32 GB 实测（2 万步冒烟，`learning_starts=1000`）：MPS 约 385 fps，CPU 约 143 fps；按此估算 1M 步在 MPS 上约 40~45 分钟（前 10 万步不训练网络，会更快）。

内存：`optimize_memory_usage: False` 时回放缓冲区同时存 obs 和 next_obs，约 `100000 × 84×84×4 字节 × 2 ≈ 5.6 GB`。内存吃紧就调小 `buffer_size`。

在超参数优化方面，我建议重点关注以下 3 个超参数：
- `learning_rate` (学习率)
- `buffer_size` (经验回放缓冲区大小)
- `batch_size` (批大小)

作为良好的实践，你需要**阅读文档以了解每个超参数的具体作用**：https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html#parameters

2. 我们开始训练并将模型保存到 `logs` 文件夹 📁 中

- 在 `--algo` 后指定算法，在 `-f` 后指定保存模型的目录，并在 `-c` 后指定超参数配置文件的路径。

**练习**（下面是待填空的代码，答案见下一节「解决方案」）：

```python
!python -m rl_zoo3.train --algo ________ --env SpaceInvadersNoFrameskip-v4  -f _________  -c _________
```

#### 解决方案

### 先跑一次冒烟测试

`learning_starts: 100000` 意味着前 10 万步只收集经验、不训练网络，MPS 上的训练代码要到第 10 万步才第一次执行。先用 2 万步、`learning_starts=1000` 跑一遍，几分钟内就能确认整条链路（ALE → AtariWrapper → MPS 训练 → 保存模型）没问题。结果存到 `logs/smoke/`，不影响后面的 `logs/`。

In [ ]:
!python -m rl_zoo3.train --algo dqn --env SpaceInvadersNoFrameskip-v4 -f logs/smoke/ -c dqn.yml -n 20000 -params learning_starts:1000 --device {DEVICE}

### 正式训练

训练时间较长，也可以把命令复制到终端里跑（在 `notebooks/unit3/` 下，把 `{sys.executable}` 换成 `uv run python`，`{DEVICE}` 换成 `mps`）。

In [ ]:
!python -m rl_zoo3.train --algo dqn  --env SpaceInvadersNoFrameskip-v4 -f logs/ -c dqn.yml --device {DEVICE}

## 让我们评估一下我们的智能体 👀
- RL-Baselines3-Zoo 提供了 `enjoy.py`，这是一个用于评估我们智能体的 Python 脚本。在大多数 RL 库中，我们都将评估脚本称为 `enjoy.py`。
- 让我们对其进行 5000 个时间步长的评估 🔥

**练习**（下面是待填空的代码，答案见下一节「解决方案」）：

```python
!python -m rl_zoo3.enjoy  --algo dqn  --env SpaceInvadersNoFrameskip-v4  --no-render  --n-timesteps _________  --folder logs/
```

#### 解决方案

In [ ]:
!python -m rl_zoo3.enjoy  --algo dqn  --env SpaceInvadersNoFrameskip-v4  --no-render  --n-timesteps 5000  --folder logs/

## 录制回放 🎥

`rl_zoo3.record_video` 用 `render_mode="rgb_array"` 逐帧渲染，经 SB3 的 `VecVideoRecorder`（moviepy）写成 mp4，这里存到 `logs/videos/` 并内联播放。Colab 版里这一步由 `push_to_hub` 顺带完成。

In [ ]:
!python -m rl_zoo3.record_video --algo dqn --env SpaceInvadersNoFrameskip-v4 -n 1000 -f logs/ -o logs/videos/
Video(max(glob.glob("logs/videos/*.mp4"), key=os.path.getmtime), embed=True)

## 上传到 Hub（本地版已省略）

本地版去掉了 `notebook_login` / `rl_zoo3.push_to_hub` 流程。想参加认证、把模型推到 [排行榜](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard)，用 Colab 版 [unit3.ipynb](unit3.ipynb) 里的上传步骤即可：`push_to_hub` 会评估智能体、录回放、生成模型卡片并推送。认证要求 `mean_reward - std_reward >= 200`。

## 加载一个强大的已训练模型 🔥
- Stable-Baselines3 团队在 Hub 上上传了 **150 多个训练有素的深度强化学习智能体**。

你可以在这里找到它们：👉 https://huggingface.co/sb3

一些例子：
- 🚀 经典 Asteroids: https://huggingface.co/sb3/dqn-AsteroidsNoFrameskip-v4
- 🛸 Beam Rider: https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4
- 🧱 Breakout: https://huggingface.co/sb3/dqn-BreakoutNoFrameskip-v4
- 🏃‍♂️ Road Runner: https://huggingface.co/sb3/dqn-RoadRunnerNoFrameskip-v4

让我们加载一个玩 Beam Rider 的智能体：https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4/resolve/main/replay.mp4" type="video/mp4"></video>

1. 我们使用 `rl_zoo3.load_from_hub` 下载模型，并将其放入一个可以称为 `rl_trained` 的新文件夹中

In [ ]:
# Download model and save it into the rl_trained/ folder
# rl_zoo3 用 requests 的 HTTPError 判断「仓库里没有 vec_normalize.pkl」；
# huggingface-hub 1.x 改用 httpx，缺文件时抛 EntryNotFoundError，命令行版会直接崩。
# 这里在内核里调用同一个函数，把它捕获的异常类型换掉。rl_zoo3 修复后可改回原版命令。
import rl_zoo3.load_from_hub as zoo_hub
from huggingface_hub.errors import EntryNotFoundError
from huggingface_sb3 import EnvironmentName

zoo_hub.HTTPError = EntryNotFoundError
zoo_hub.download_from_hub("dqn", EnvironmentName("BeamRiderNoFrameskip-v4"), exp_id=0, folder="rl_trained/", organization="sb3")

2. 让我们对它进行 5000 个时间步的评估

In [ ]:
!python -m rl_zoo3.enjoy --algo dqn --env BeamRiderNoFrameskip-v4 -n 5000  -f rl_trained/ --no-render

为什么不尝试训练你自己的 **Deep Q-Learning 智能体玩 BeamRiderNoFrameskip-v4 呢？🏆**

如果你想尝试，请查看 https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4#hyperparameters **在模型卡片中，有已训练智能体的超参数。**

但是寻找超参数可能是一项艰巨的任务。幸运的是，我们将在下一个单元中看到如何**使用 Optuna 优化超参数 🔥。**

## 一些额外的挑战 🏆
最好的学习方法**就是自己动手尝试**！

在 [排行榜](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard) 中你会找到你的智能体。你能名列前茅吗？

这里是可以尝试用来训练智能体的环境列表：
- BeamRiderNoFrameskip-v4
- BreakoutNoFrameskip-v4
- EnduroNoFrameskip-v4
- PongNoFrameskip-v4

此外，**如果你想自己学习实现 Deep Q-Learning**，你绝对应该看看 CleanRL 的实现：https://github.com/vwxyzjn/cleanrl/blob/master/cleanrl/dqn_atari.py

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/atari-envs.gif" alt="Environments"/>

________________________________________________________________________
恭喜你完成本章！

如果你对所有这些元素仍然感到困惑……这完全正常！**对于我和所有学习强化学习的人来说都是一样的。**

在继续并尝试额外挑战之前，花点时间**真正掌握这些材料**。掌握这些元素并打下坚实的基础非常重要。

在下一个单元中，**我们将学习 [Optuna](https://optuna.org/)**。深度强化学习中最关键的任务之一是找到一组优秀的训练超参数。而 Optuna 是一个可以帮助您自动进行搜索的库。

### 这是一个为您而建的课程 👷🏿‍♀️

最后，我们希望根据您的反馈不断改进和更新课程。如果您有一些建议，请填写此表单 👉 https://forms.gle/3HgA7bEHwAmmLfwh9

我们一直在努力改进我们的教程，因此**如果您在此 Notebook 中发现任何问题**，请在 [Github 仓库上提交 Issue](https://github.com/huggingface/deep-rl-class/issues)。

期待在 Bonus unit 2 与你相见！🔥

### 保持学习，保持优秀 🤗